## Install missing libraries

In [ ]:
#!pip install pymysql & pip install sqlalchemy

In [ ]:

import pandas as pd
from sqlalchemy import create_engine

import sklearn
import datetime
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn import metrics
from sklearn.metrics import (
    accuracy_score, precision_score, 
    recall_score, mean_absolute_error, root_mean_squared_error, r2_score, 
    f1_score, confusion_matrix, ConfusionMatrixDisplay, mean_absolute_percentage_error)
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.eval_measures import mse
import pickle


In [ ]:
random_variable = 42

In [ ]:
def evaluate_model(y_true, y_pred, display_labels=['true', 'false']):
    cnf_matrix = confusion_matrix(y_true, y_pred, labels=[True, False])
    disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matrix.T, 
                                 display_labels=display_labels)

    # Create a figure and plot
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, cmap='Blues')
    plt.xlabel('Actual Condition')  # This will now be on the x-axis
    plt.ylabel('Predicted Condition')  # This will now be on the y-axis
    plt.title('Confusion Matrix')
    plt.show()

    # Add precision, recall, and F1-score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f'Accuracy: {accuracy:.2f}')
    print(f'Precision: {precision:.2f}')
    print(f'Recall: {recall:.2f}')
    print(f'F1-score: {f1:.2f}')


In [ ]:
user = "readonly"
password = ""
db = ""
host = ""

In [ ]:
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:3306/{db}")

### Get relevant columns for purchase orders

In [ ]:
purchase_orders = pd.read_sql("""
    Select name, 
            supplier, 
            schedule_date,                 
            total_qty, 
            transaction_date, 
            set_warehouse
    FROM `tabPurchase Order`
    """, engine)

In [ ]:
purchase_orders.head()

In [ ]:
purchase_orders.shape

### Get relevant columns from Table "Purchase Order Item"

In [ ]:
purchase_order_items = pd.read_sql(
                """
                Select 
                    name, 
                    parent, 
                    item_name, 
                    schedule_date,
                    rate,         
                    qty, 
                    warehouse                                    
                FROM `tabPurchase Order Item`
                """, 
                engine)

In [ ]:
purchase_order_items.head()

Get a specific purchase order

In [ ]:
purchase_order_items[purchase_order_items["parent"] == "PUR-ORD-2026-09-00001"]

### Combine Data in Pandas

In [ ]:
data = pd.merge(purchase_orders, purchase_order_items, left_on="name", right_on="parent")

In [ ]:
data.head()

### Better Strategy: Combine Data with SQL

In [ ]:
data_sql = pd.read_sql("""
   Select 
    po.name, 
    po.supplier,            
    po.total_qty, 
    po.transaction_date, 
    po.set_warehouse,
    po_items.item_name, 
    po_items.schedule_date,
    po_items.rate,         
    po_items.qty, 
    po_items.warehouse 
FROM `tabPurchase Order` po
LEFT JOIN `tabPurchase Order Item` po_items  
ON po.name = po_items.parent;
""",
engine)

In [ ]:
data_sql.head()

### Convert Date

In [ ]:
data_sql["transaction_date"] = pd.to_datetime(data_sql["transaction_date"])

In [ ]:
data_sql["schedule_date"] = pd.to_datetime(data_sql["schedule_date"])

In [ ]:
data_sql.head()

### Purchase Receipts

In [ ]:
purchase_receipts = pd.read_sql(
    """
    Select 
        name, 
        posting_date 
    FROM `tabPurchase Receipt`
    """, 
    engine)

In [ ]:
purchase_receipts.head()

In [ ]:
data = pd.read_sql("""
    Select 
    po.name, 
    po.supplier,            
    po.total_qty, 
    po.transaction_date, 
    po.set_warehouse,
    po_items.item_name, 
    po_items.schedule_date,
    po_items.rate,         
    po_items.qty, 
    po_items.warehouse,
    po_receipt_item.parent as receipt_item_parent	
FROM `tabPurchase Order` po
LEFT JOIN `tabPurchase Order Item` po_items  
	ON po.name = po_items.parent
LEFT JOIN `tabPurchase Receipt Item` po_receipt_item
	ON po.name = po_receipt_item.purchase_order;
;
""",
engine)

In [ ]:
data.head()

In [ ]:
data = pd.read_sql("""
     Select 
    po.name, 
    po.supplier,            
    po.total_qty, 
    po.transaction_date, 
    po.set_warehouse,
    po_items.item_name, 
    po_items.schedule_date,
    po_items.rate,         
    po_items.qty, 
    po_items.warehouse,
    po_receipt_item.parent as receipt_item_parent,
    po_receipt.posting_date as delivered_date
FROM `tabPurchase Order` po
LEFT JOIN `tabPurchase Order Item` po_items  
	ON po.name = po_items.parent
LEFT JOIN `tabPurchase Receipt Item` po_receipt_item
	ON po.name = po_receipt_item.purchase_order
LEFT JOIN `tabPurchase Receipt` po_receipt
	ON po_receipt_item.parent =  po_receipt.name 
;
""",
engine)

In [ ]:
data.head()

In [ ]:
data.shape

Convert timestamps

In [ ]:
data["delivered_date"] = pd.to_datetime(data["delivered_date"])
data["schedule_date"] = pd.to_datetime(data["schedule_date"])

Delete Rows with empty delivered or scheduled date

In [ ]:
data.dropna(subset=["delivered_date", "schedule_date"], inplace=True)

Calculate Days late

In [ ]:
data["days_late"] = (data["delivered_date"] - data["schedule_date"]).dt.days

In [ ]:
data["days_late"].mean()

In [ ]:
data["days_late"].hist()

Calculate Late vs. Not Late

In [ ]:
data["late"] = data["days_late"] > 0

In [ ]:
data["late"] .mean()

### Create Dummy Encodings

#### Product Category

In [ ]:
d_product_categories = pd.get_dummies(data["item_name"], prefix="c").astype(int)

### Warehouse

In [ ]:
d_warehouse = pd.get_dummies(data["set_warehouse"], prefix="w").astype(int)

#### Supplier

In [ ]:
d_supplier = pd.get_dummies(data["supplier"], prefix="s").astype(int)

### Month

In [ ]:
d_month = pd.get_dummies(data["schedule_date"].apply(lambda x: x.month), prefix="m").astype(int) 

#### Join all data together

In [ ]:
full_dataset = pd.concat(
    [ 
        data,
        d_product_categories,
        d_warehouse,
        d_supplier,
        d_month               
    ], 
    axis=1
)

### Split Dataset

In [ ]:
print(data["schedule_date"].min(), data["schedule_date"].max())

In [ ]:
first_date_test = pd.to_datetime("2025-01-01")

In [ ]:
train = full_dataset[full_dataset["schedule_date"] < first_date_test]

In [ ]:
test = full_dataset[full_dataset["schedule_date"] >= first_date_test]

In [ ]:
features = (["total_qty"] 
            + list(d_product_categories.columns) 
            + list(d_warehouse.columns) 
            + list(d_supplier.columns) 
            + list(d_month.columns) 
)

In [ ]:
X_train = train[features]
y_train = train["late"]
X_test = test[features]
y_test = test["late"]

## Building a Model for Late vs. Not Late

### Log-Regression

In [ ]:
logreg = LogisticRegression(random_state = random_variable , max_iter=10)

In [ ]:
logreg.fit(X_train, y_train)

In [ ]:
log_reg_prediction_test = logreg.predict(X_test)

In [ ]:
metrics.confusion_matrix(y_test, log_reg_prediction_test, labels=[True, False]).T

In [ ]:
evaluate_model(y_test, log_reg_prediction_test)

### Decision Tree (Depth-Level 3)

In [ ]:
clf_depth_3 = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=3)

In [ ]:
clf_depth_3.fit(X_train, y_train)

In [ ]:
tree_3_predictions = clf_depth_3.predict(X_test)

In [ ]:
evaluate_model(y_test, tree_3_predictions)

### Decision Tree (Depth-Level 10)

In [ ]:
clf_depth_10 = tree.DecisionTreeClassifier(random_state=random_variable, max_depth=10)

In [ ]:
clf_depth_10.fit(X_train, y_train)

In [ ]:
tree_10_predictions = clf_depth_10.predict(X_test)

In [ ]:
evaluate_model(y_test, tree_10_predictions)


### Random Forest

In [ ]:
rf_500 = RandomForestClassifier(random_state = random_variable, n_estimators = 500)

In [ ]:
rf_500.fit(X_train, y_train)

In [ ]:
predictions_rf_500 = rf_500.predict(X_test)

In [ ]:
evaluate_model(y_test, predictions_rf_500)

### Gradient Boosted Trees

In [ ]:
gbm_500 = GradientBoostingClassifier(random_state = random_variable, n_estimators = 500)

In [ ]:
gbm_500.fit(X_train, y_train)

In [ ]:
predictions_gbm_500 = gbm_500.predict(X_test)

In [ ]:
evaluate_model(y_test, predictions_gbm_500)

### Compare Model results

In [ ]:
pd.DataFrame({
    'Score': ["Accuracy", "Precision", "Recall", "F1"],
    'tree_10': [
        round(accuracy_score(y_test, tree_10_predictions), 2), 
        round(precision_score(y_test, tree_10_predictions), 2),
        round(recall_score(y_test, tree_10_predictions), 2),
        round(f1_score(y_test, tree_10_predictions), 2)

    ],
    'rf_500': [
        round(accuracy_score(y_test, predictions_rf_500), 2), 
        round(precision_score(y_test, predictions_rf_500), 2),
        round(recall_score(y_test, predictions_rf_500), 2),
        round(f1_score(y_test, predictions_rf_500), 2)
    ],
    'gbm_500': [
        round(accuracy_score(y_test, predictions_gbm_500), 2), 
        round(precision_score(y_test, predictions_gbm_500), 2),
        round(recall_score(y_test, predictions_gbm_500), 2),
        round(f1_score(y_test, predictions_gbm_500), 2)
    ],
})

# Building Model for days late

## Regression models for predicting days late


In [ ]:
y_train_reg = train["days_late"]
y_test_reg = test["days_late"]


In [ ]:

days_late_ols = LinearRegression().fit(X_train, y_train_reg)
predictions_days_late_ols = days_late_ols.predict(X_test)


#### Decision Tree Depth 3

In [ ]:
tree_reg_depth_3 = tree.DecisionTreeRegressor(random_state=random_variable, max_depth=3)
tree_reg_depth_3.fit(X_train, y_train_reg)
predictions_days_late_tree_3 = tree_reg_depth_3.predict(X_test)


#### Decision Tree Depth 10


In [ ]:
tree_reg_depth_10 = tree.DecisionTreeRegressor(random_state=random_variable, max_depth=10)
tree_reg_depth_10.fit(X_train, y_train_reg)
predictions_days_late_tree_10 = tree_reg_depth_10.predict(X_test)


#### Random Forest (500 trees)

In [ ]:
rf_reg_500 = RandomForestRegressor(random_state=random_variable, n_estimators=500)
rf_reg_500.fit(X_train, y_train_reg)
predictions_days_late_rf_500 = rf_reg_500.predict(X_test)

#### Gradient Boosting (500 trees)

In [ ]:
gbm_reg_500 = GradientBoostingRegressor(random_state=random_variable, n_estimators=500)
gbm_reg_500.fit(X_train, y_train_reg)
predictions_days_late_gbm_500 = gbm_reg_500.predict(X_test)

#### Comparison of Models

WAPE (Weighted Absolute Percentage Error)
= total absolute error divided by total actual volume.
In plain terms: “overall error size relative to total true days late.”
Example: WAPE = 25% means your total prediction error is about 25% of total actual lateness.

In [ ]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denominator


### WAPE formula

$$\text{WAPE} = \frac{\sum_{i=1}^{n} |y_i - \hat{y}_i|}{\sum_{i=1}^{n} |y_i|} \times 100\%$$

where $y_i$ is actual `days_late` and $\hat{y}_i$ is predicted `days_late`.


### Root Mean Squared Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

RMSE measures the typical prediction error size in the same unit as the target (`days_late`).
Because errors are squared before averaging, larger errors are penalized more strongly. Lower RMSE is better.


In [ ]:
pd.DataFrame({
    "Score": ["MAE", "RMSE", "WAPE_%"],
    "ols": [
        round(mean_absolute_error(y_test_reg, predictions_days_late_ols), 2),
        round(root_mean_squared_error(y_test_reg, predictions_days_late_ols), 2),
        round(100 * wape(y_test_reg, predictions_days_late_ols), 2)
        
    ],
    "tree_3": [
        round(mean_absolute_error(y_test_reg, predictions_days_late_tree_3), 2),
        round(root_mean_squared_error(y_test_reg, predictions_days_late_tree_3), 2),
        round(100 * wape(y_test_reg, predictions_days_late_tree_3), 2)        
    ],
    "tree_10": [
        round(mean_absolute_error(y_test_reg, predictions_days_late_tree_10), 2),
        round(root_mean_squared_error(y_test_reg, predictions_days_late_tree_10), 2),
        round(100 * wape(y_test_reg, predictions_days_late_tree_10), 2)        
    ],
    "rf_500": [
        round(mean_absolute_error(y_test_reg, predictions_days_late_rf_500), 2),
        round(root_mean_squared_error(y_test_reg, predictions_days_late_rf_500), 2),
        round(100 * wape(y_test_reg, predictions_days_late_rf_500), 2)        
    ],
    "gbm_500": [
        round(mean_absolute_error(y_test_reg, predictions_days_late_gbm_500), 2),
        round(root_mean_squared_error(y_test_reg, predictions_days_late_gbm_500), 2),
        round(100 * wape(y_test_reg, predictions_days_late_gbm_500), 2)        
    ],
})


### Save Model

In [ ]:
with open("gbm_reg_500.pkl", "wb") as file:
    pickle.dump(gbm_reg_500, file)

### Load Model

In [ ]:
    
with open("gbm_reg_500.pkl", "rb") as file:
    gbm_500_regression = pickle.load(file)